# Proyecto final – Preparación del modelo y prueba de la API con Cloudflare Tunnel
**Proyecto:** Asistente de reclutamiento del restaurante El Sauce · **Alumna:** Juana Paola Puente Huerta

Este cuaderno tiene dos partes:
1. **Convertir el modelo de visión a TensorFlow Lite.** El modelo Keras pesa mucho para la nube; la versión Lite
   da las mismas predicciones y carga mucho más rápido en Streamlit Community Cloud.
2. **Probar la API REST (Flask) desde internet con Cloudflare Tunnel**, como evidencia de que la API funciona
   como servicio independiente de la interfaz.

## Parte 1. Convertir el modelo a TensorFlow Lite

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json, numpy as np, tensorflow as tf, keras
from PIL import Image, ImageOps

RUTA = '/content/drive/MyDrive/ChatbotRH'
RUTA_MODELO = os.path.join(RUTA, 'modelo')

modelo = keras.models.load_model(os.path.join(RUTA_MODELO, 'modelo_documentos.keras'))
CLASES = json.load(open(os.path.join(RUTA_MODELO, 'clases.json')))
print("Modelo Keras cargado. Clases:", CLASES)

In [ ]:
# Se exporta como SavedModel (modo inferencia: el aumento de datos queda desactivado) y se convierte a TFLite
modelo.export('/content/modelo_saved')
convertidor = tf.lite.TFLiteConverter.from_saved_model('/content/modelo_saved')
modelo_tflite = convertidor.convert()

ruta_tflite = os.path.join(RUTA_MODELO, 'modelo_documentos.tflite')
with open(ruta_tflite, 'wb') as f:
    f.write(modelo_tflite)
print(f"Keras: {os.path.getsize(os.path.join(RUTA_MODELO, 'modelo_documentos.keras')) / 1e6:.1f} MB")
print(f"TFLite: {os.path.getsize(ruta_tflite) / 1e6:.1f} MB  ->  guardado en {ruta_tflite}")

### Verificación: el modelo Lite debe dar las mismas predicciones que el original

In [ ]:
interprete = tf.lite.Interpreter(model_path=ruta_tflite)
interprete.allocate_tensors()
ent = interprete.get_input_details()[0]['index']
sal = interprete.get_output_details()[0]['index']

def cargar(ruta):
    img = ImageOps.exif_transpose(Image.open(ruta)).convert('RGB')
    return np.asarray(ImageOps.pad(img, (224, 224), color=(0, 0, 0)), dtype=np.float32)[None, ...]

RUTA_PRUEBAS = os.path.join(RUTA, 'pruebas')
for archivo in sorted(os.listdir(RUTA_PRUEBAS)):
    if archivo.lower().endswith('.pdf'):
        continue
    x = cargar(os.path.join(RUTA_PRUEBAS, archivo))
    p_keras = modelo.predict(x, verbose=0)[0]
    interprete.set_tensor(ent, x); interprete.invoke()
    p_lite = interprete.get_tensor(sal)[0]
    print(f"{archivo:<28} Keras: {CLASES[p_keras.argmax()]:<12} {p_keras.max():.3f} | "
          f"TFLite: {CLASES[p_lite.argmax()]:<12} {p_lite.max():.3f} | diferencia máx: {np.abs(p_keras - p_lite).max():.5f}")

Si las clases coinciden y la diferencia es muy pequeña (menor a 0.01), la conversión es correcta.

**Siguiente paso:** descarga `modelo_documentos.tflite` y `clases.json` de la carpeta `ChatbotRH/modelo`
de tu Drive y súbelos a la carpeta `proyecto_final/modelo/` de tu repositorio en GitHub.

## Parte 2. Probar la API REST con Cloudflare Tunnel
Se descarga el código desde GitHub, se ejecuta la API Flask en este cuaderno y Cloudflare Tunnel
le asigna una **URL pública temporal** para poder llamarla desde cualquier lugar.

> La URL de Cloudflare deja de funcionar cuando se cierra este cuaderno. La URL permanente de la aplicación
> es la de Streamlit Community Cloud; esta prueba sirve para demostrar la API como servicio independiente.

In [ ]:
%cd /content
!rm -rf Chatbot-RH
!git clone -q https://github.com/paolaphuerta/Chatbot-RH.git
%cd /content/Chatbot-RH/proyecto_final
!pip install -q flask nltk pillow-heif pymupdf ai-edge-litert
!ls modelo

In [ ]:
# Inicia la API Flask en segundo plano (puerto 5000)
import subprocess, time, requests
api = subprocess.Popen(['python', '-m', 'api.servidor'], stdout=open('/content/api.log', 'w'), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        if requests.get('http://localhost:5000/api/salud', timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)
print(requests.get('http://localhost:5000/api/salud').json())

In [ ]:
# Descarga cloudflared y abre un túnel público hacia la API
!wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x /content/cloudflared
import re
tunel = subprocess.Popen(['/content/cloudflared', 'tunnel', '--url', 'http://localhost:5000', '--no-autoupdate'],
                         stdout=open('/content/tunel.log', 'w'), stderr=subprocess.STDOUT)
URL = None
for _ in range(60):
    time.sleep(1)
    m = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', open('/content/tunel.log').read())
    if m:
        URL = m.group(0); break
print("URL pública de la API:", URL)

### Pruebas de los servicios de la API a través de la URL pública

In [ ]:
import json
def ver(r):
    print(r.status_code, json.dumps(r.json(), ensure_ascii=False, indent=2)[:1200], '\n')

time.sleep(5)   # el túnel tarda unos segundos en estar listo
print("GET /api/salud"); ver(requests.get(URL + '/api/salud', timeout=30))
print("POST /api/intencion"); ver(requests.post(URL + '/api/intencion', json={'mensaje': 'cuanto pagan d mesra'}, timeout=30))

In [ ]:
# Clasificación de un documento (usa una imagen de tu carpeta de pruebas)
ruta_img = [os.path.join(RUTA_PRUEBAS, f) for f in sorted(os.listdir(RUTA_PRUEBAS)) if f.startswith('ine')][0]
with open(ruta_img, 'rb') as f:
    print("POST /api/documento"); ver(requests.post(URL + '/api/documento', files={'archivo': (os.path.basename(ruta_img), f)}, timeout=60))

In [ ]:
# Conversación con el agente a través de la API
sesion = 'prueba-colab'
for m in ['hola', 'me interesa ser runner', '22', 'si']:
    r = requests.post(URL + '/api/agente/mensaje', json={'sesion': sesion, 'mensaje': m}, timeout=30).json()
    print(f"Usuario: {m}\n  [{r['capa']} · {r['accion']} · plan: {r['plan'][:3]}]\nBot: {r['respuesta']}\n")

In [ ]:
# Manejo de errores: la API responde con un mensaje claro en lugar de fallar
print("Sin mensaje:"); ver(requests.post(URL + '/api/agente/mensaje', json={'sesion': 'x'}, timeout=30))
print("Archivo no permitido:"); ver(requests.post(URL + '/api/documento', files={'archivo': ('notas.txt', b'hola')}, timeout=30))
print("Métricas de operación:"); ver(requests.get(URL + '/api/metricas', timeout=30))

In [ ]:
# Al terminar, cierra el túnel y la API
tunel.terminate(); api.terminate()